In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/Users/joelmathooko/Desktop/PROJECTS/Jumia Project/Excel_jumia_dataset.csv')

df.head()

,Product,Current price,old price,Discount,Review,Ratingd
0,115 Piece Set Of Multifunctional Precision Sc...,KSh 950,"KSh 1,525",38%,-2.0,4.5 out of 5
1,Metal Decorative Hooks Key Hangers Entryway Wa...,KSh 527,KSh 999,47%,-14.0,4.1 out of 5
2,Portable Mini Cordless Car Vacuum Cleaner - Blue,"KSh 2,199","KSh 2,923",25%,-24.0,4.6 out of 5
3,Weighing Scale Digital Bathroom Body Fat Scale...,"KSh 1,580","KSh 2,499",37%,-7.0,4.7 out of 5
4,Portable Home Small Air Humidifier 3-Speed Fan...,"KSh 1,740","KSh 2,356",26%,-5.0,4.8 out of 5


In [3]:
df.shape

(115, 6)

In [4]:
df.columns

Index(['Product', 'Current price', 'old price', 'Discount', 'Review',
       'Ratingd'],
      dtype='object')

In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [6]:
df.columns

Index(['product', 'current_price', 'old_price', 'discount', 'review',
       'ratingd'],
      dtype='object')

In [7]:
df = df.rename(columns={"ratingd": "rating"})

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product        115 non-null    object 
 1   current_price  115 non-null    object 
 2   old_price      115 non-null    object 
 3   discount       115 non-null    object 
 4   review         57 non-null     float64
 5   rating         57 non-null     object 
dtypes: float64(1), object(5)
memory usage: 5.5+ KB


In [9]:
df.isna().sum()

product           0
current_price     0
old_price         0
discount          0
review           58
rating           58
dtype: int64

In [10]:
df["total_rating"] = (
    df["rating"]
    .str.extract(r"out of\s+(\d+)", expand=False)
    .astype("Int64")
)

df["rating"] = (
    df["rating"]
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

In [11]:
df[["rating", "total_rating"]].head()

,rating,total_rating
0,4.5,5
1,4.1,5
2,4.6,5
3,4.7,5
4,4.8,5


In [12]:
for col in ["current_price", "old_price"]:
    prices = df[col].str.replace("KSh ", "").str.replace(",", "")
    
    df[f"{col}_min"] = prices.str.split(" - ").str[0].astype(float)
    df[f"{col}_max"] = prices.str.split(" - ").str[-1].astype(float)

In [13]:
df["current_price_mid"] = (
    df["current_price_min"] + df["current_price_max"]
) / 2

df["old_price_mid"] = (
    df["old_price_min"] + df["old_price_max"]
) / 2

In [14]:
print(df.columns.tolist())

['product', 'current_price', 'old_price', 'discount', 'review', 'rating', 'total_rating', 'current_price_min', 'current_price_max', 'old_price_min', 'old_price_max', 'current_price_mid', 'old_price_mid']


In [15]:
df[
    [
        "current_price_min",
        "current_price_mid",
        "old_price_min",
        "old_price_mid"
    ]
].head()

,current_price_min,current_price_mid,old_price_min,old_price_mid
0,950.0,950.0,1525.0,1525.0
1,527.0,527.0,999.0,999.0
2,2199.0,2199.0,2923.0,2923.0
3,1580.0,1580.0,2499.0,2499.0
4,1740.0,1740.0,2356.0,2356.0


In [16]:
(
    (df["current_price_min"] > df["current_price_mid"]).sum(),
    (df["old_price_min"] > df["old_price_mid"]).sum()
)

(np.int64(0), np.int64(0))

In [17]:
df = df.drop(columns=["current_price", "old_price"])

In [18]:
df["discount"] = (
    df["discount"]
    .str.replace("%", "")
    .astype(float) / 100
)

In [19]:
df.select_dtypes(include="number").lt(0).sum()

discount              0
review               57
rating                0
total_rating          0
current_price_min     0
current_price_max     0
old_price_min         0
old_price_max         0
current_price_mid     0
old_price_mid         0
dtype: Int64

In [20]:
df["review"] = df["review"].abs()

In [21]:
df.select_dtypes(include="number").lt(0).sum()

discount             0
review               0
rating               0
total_rating         0
current_price_min    0
current_price_max    0
old_price_min        0
old_price_max        0
current_price_mid    0
old_price_mid        0
dtype: Int64

In [22]:
df.isna().sum()

product               0
discount              0
review               58
rating               58
total_rating         58
current_price_min     0
current_price_max     0
old_price_min         0
old_price_max         0
current_price_mid     0
old_price_mid         0
dtype: int64

In [23]:
df.duplicated().sum()

np.int64(3)

In [24]:
df[df.duplicated(keep=False)].sort_values("product")

,product,discount,review,rating,total_rating,current_price_min,current_price_max,old_price_min,old_price_max,current_price_mid,old_price_mid
95,6 Layers Steel Pipe Assembling Dustproof Stora...,0.47,NaN,NaN,<NA>,899.0,899.0,1699.0,1699.0,899.0,1699.0
96,6 Layers Steel Pipe Assembling Dustproof Stora...,0.47,NaN,NaN,<NA>,899.0,899.0,1699.0,1699.0,899.0,1699.0
42,"Balloon Insert, Birthday Party Balloon Set, PU...",0.42,NaN,NaN,<NA>,610.0,610.0,1060.0,1060.0,610.0,1060.0
43,"Balloon Insert, Birthday Party Balloon Set, PU...",0.42,NaN,NaN,<NA>,610.0,610.0,1060.0,1060.0,610.0,1060.0
84,"Balloon Insert, Birthday Party Balloon Set, PU...",0.42,NaN,NaN,<NA>,610.0,610.0,1060.0,1060.0,610.0,1060.0


In [25]:
df = df.drop_duplicates()

In [26]:
df = df.reset_index(drop=True)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product            112 non-null    object 
 1   discount           112 non-null    float64
 2   review             57 non-null     float64
 3   rating             57 non-null     float64
 4   total_rating       57 non-null     Int64  
 5   current_price_min  112 non-null    float64
 6   current_price_max  112 non-null    float64
 7   old_price_min      112 non-null    float64
 8   old_price_max      112 non-null    float64
 9   current_price_mid  112 non-null    float64
 10  old_price_mid      112 non-null    float64
dtypes: Int64(1), float64(9), object(1)
memory usage: 9.9+ KB


In [28]:
df.isna().sum()

product               0
discount              0
review               55
rating               55
total_rating         55
current_price_min     0
current_price_max     0
old_price_min         0
old_price_max         0
current_price_mid     0
old_price_mid         0
dtype: int64

In [29]:
df.select_dtypes(include="number").lt(0).sum()

discount             0
review               0
rating               0
total_rating         0
current_price_min    0
current_price_max    0
old_price_min        0
old_price_max        0
current_price_mid    0
old_price_mid        0
dtype: Int64

In [30]:
print(df.columns.tolist())

['product', 'discount', 'review', 'rating', 'total_rating', 'current_price_min', 'current_price_max', 'old_price_min', 'old_price_max', 'current_price_mid', 'old_price_mid']


In [31]:
df["discount_amount"] = (
    df["old_price_mid"] - df["current_price_mid"]
)

In [32]:
df[[
    "product",
    "current_price_mid",
    "old_price_mid",
    "discount_amount",
    "discount"
]].head(10)

,product,current_price_mid,old_price_mid,discount_amount,discount
0,115 Piece Set Of Multifunctional Precision Sc...,950.0,1525.0,575.0,0.38
1,Metal Decorative Hooks Key Hangers Entryway Wa...,527.0,999.0,472.0,0.47
2,Portable Mini Cordless Car Vacuum Cleaner - Blue,2199.0,2923.0,724.0,0.25
3,Weighing Scale Digital Bathroom Body Fat Scale...,1580.0,2499.0,919.0,0.37
4,Portable Home Small Air Humidifier 3-Speed Fan...,1740.0,2356.0,616.0,0.26
5,220V 60W Electric Soldering Iron Kits With Too...,2999.0,3290.0,291.0,0.09
6,137 Pieces Cake Decorating Tool Set Baking Sup...,2319.0,3032.0,713.0,0.24
7,Desk Foldable Fan Adjustable Fan Strong Wind 3...,988.0,1580.0,592.0,0.37
8,LASA FOLDING TABLE SERVING STAND,1274.0,2800.0,1526.0,0.55
9,13 In 1 Home Repair Tools Box Kit Set,1600.0,2929.0,1329.0,0.45


In [33]:
df["calculated_discount_pct"] = (
    df["discount_amount"] / df["old_price_mid"]
)

df["discount_difference"] = (
    df["calculated_discount_pct"] - df["discount"]
)

df["discount_difference"].describe()

count    112.000000
mean      -0.000627
std        0.005240
min       -0.046667
25%       -0.002735
50%        0.000000
75%        0.002015
max        0.004925
Name: discount_difference, dtype: float64

In [34]:
df.loc[
    df["discount_difference"].abs() > 0.02,
    [
        "product",
        "current_price_mid",
        "old_price_mid",
        "discount_amount",
        "discount",
        "calculated_discount_pct",
        "discount_difference"
    ]
]

,product,current_price_mid,old_price_mid,discount_amount,discount,calculated_discount_pct,discount_difference
38,"1/2/3 Seater Elastic Sofa Cover,Living Room/Ho...",1800.0,2700.0,900.0,0.38,0.333333,-0.046667


In [35]:
df["rating_category"] = np.select(
    [
        df["rating"] < 3,
        df["rating"] <= 4.5,
        df["rating"] > 4.5
    ],
    [
        "Poor",
        "Average",
        "Excellent"
    ],
    default="Missing"
)

In [36]:
df["rating_category"].value_counts(dropna=False)

rating_category
Missing      55
Average      26
Excellent    19
Poor         12
Name: count, dtype: int64

In [37]:
df["discount_category"] = np.select(
    [
        df["discount"] < 0.20,
        df["discount"] <= 0.40,
        df["discount"] > 0.40
    ],
    [
        "Low",
        "Medium",
        "High"
    ],
    default="Unknown"
)

In [38]:
df["discount_category"].value_counts()

discount_category
High      62
Medium    32
Low       18
Name: count, dtype: int64

In [39]:
median_price = df["current_price_mid"].median()

print(median_price)

998.5


In [40]:
df["current_price_mid"].describe()

count     112.000000
mean     1186.892857
std       832.346443
min        38.000000
25%       493.000000
50%       998.500000
75%      1669.500000
max      3750.000000
Name: current_price_mid, dtype: float64

In [41]:
df["price_category"] = pd.qcut(
    df["current_price_mid"],
    q=4,
    labels=["Low", "Medium", "High", "Premium"]
)

In [42]:
df["price_category"].value_counts()

price_category
Low        28
Medium     28
High       28
Premium    28
Name: count, dtype: int64

In [43]:
df.groupby("price_category", observed=True)["current_price_mid"].agg(
    ["min", "max", "count"]
)

,min,max,count
price_category,,,
Low,38.0,475.0,28
Medium,499.0,998.0,28
High,999.0,1666.0,28
Premium,1680.0,3750.0,28


In [44]:
# All numeric columns to 2 decimal places
numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = df[numeric_columns].round(3)



In [45]:
df[["calculated_discount_pct", "discount_difference"]]

,calculated_discount_pct,discount_difference
0,0.377,-0.003
1,0.472,0.002
2,0.248,-0.002
3,0.368,-0.002
4,0.261,0.001
...,...,...
107,0.024,0.004
108,0.017,-0.003
109,0.640,0.000
110,0.500,0.000


In [46]:
df.to_csv("jumia_products_cleaned.csv", index=False)

In [47]:
!pip install sqlalchemy psycopg2-binary

In [48]:
from sqlalchemy import create_engine, text

In [49]:
from db_connection import engine

In [50]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT current_user, current_database()")
    )
    print(result.fetchone())

('joelmathooko', 'jumia_products')


In [51]:
df.to_sql(
    "products",
    engine,
    if_exists="replace",
    index=False
)

112

In [52]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM products"))
    print(result.scalar())

112


In [53]:
df[["discount", "review"]].corr()

,discount,review
discount,1.000000,-0.136823
review,-0.136823,1.000000


In [54]:
df[["rating", "review"]].corr()

,rating,review
rating,1.000000,0.057209
review,0.057209,1.000000


In [55]:
df[["current_price_mid", "rating"]].corr()

,current_price_mid,rating
current_price_mid,1.00000,0.11009
rating,0.11009,1.00000


In [56]:
#Relationship	Correlation	Interpretation
#Rating ↔ Reviews	0.057	Almost no relationship
#Price ↔ Rating	0.110	Very weak positive relationship

In [57]:
from db_connection import engine
import pandas as pd

df = pd.read_sql("SELECT * FROM products", engine)

df.head()

,product,discount,review,rating,total_rating,current_price_min,current_price_max,old_price_min,old_price_max,current_price_mid,old_price_mid,discount_amount,calculated_discount_pct,discount_difference,rating_category,discount_category,price_category
0,115 Piece Set Of Multifunctional Precision Sc...,0.38,2.0,4.5,5.0,950.0,950.0,1525.0,1525.0,950.0,1525.0,575.0,0.377,-0.003,Average,Medium,Medium
1,Metal Decorative Hooks Key Hangers Entryway Wa...,0.47,14.0,4.1,5.0,527.0,527.0,999.0,999.0,527.0,999.0,472.0,0.472,0.002,Average,High,Medium
2,Portable Mini Cordless Car Vacuum Cleaner - Blue,0.25,24.0,4.6,5.0,2199.0,2199.0,2923.0,2923.0,2199.0,2923.0,724.0,0.248,-0.002,Excellent,Medium,Premium
3,Weighing Scale Digital Bathroom Body Fat Scale...,0.37,7.0,4.7,5.0,1580.0,1580.0,2499.0,2499.0,1580.0,2499.0,919.0,0.368,-0.002,Excellent,Medium,High
4,Portable Home Small Air Humidifier 3-Speed Fan...,0.26,5.0,4.8,5.0,1740.0,1740.0,2356.0,2356.0,1740.0,2356.0,616.0,0.261,0.001,Excellent,Medium,Premium


In [58]:
df.shape

(112, 17)

In [59]:
#Top 10 rated products
top_rated = (
    df.dropna(subset=["rating"])
      .sort_values(["rating", "review"], ascending=[False, False])
      [["product", "rating", "review", "discount",
        "current_price_mid", "rating_category"]]
      .head(10)
)

top_rated

,product,rating,review,discount,current_price_mid,rating_category
17,LASA Aluminum Folding Truck Hand Cart - 68kg Max,5.0,3.0,0.49,2025.0,Excellent
15,Anti-Skid Absorbent Insulation Coaster For Ho...,5.0,2.0,0.51,332.0,Excellent
16,Peacock Throw Pillow Cushion Case For Home Car,5.0,2.0,0.46,195.0,Excellent
36,Classic Black Cat Cotton Hemp Pillow Case For ...,5.0,2.0,0.53,171.0,Excellent
35,"DIY File Folder, Office Drawer File Holder, Pe...",5.0,1.0,0.40,1620.0,Excellent
79,Bedroom Simple Floor Hanging Clothes Rack Sing...,5.0,1.0,0.49,979.0,Excellent
101,"Konka Healty Electric Kettle, 24-hour Heat Pre...",5.0,1.0,0.21,3640.0,Excellent
12,40cm Gold DIY Acrylic Wall Sticker Clock,4.8,12.0,0.47,552.0,Excellent
4,Portable Home Small Air Humidifier 3-Speed Fan...,4.8,5.0,0.26,1740.0,Excellent
8,LASA FOLDING TABLE SERVING STAND,4.8,5.0,0.55,1274.0,Excellent


In [60]:
#Top 10 products with the most reviews
df["review_count"] = df["review"].abs()

top_engagement = (
    df.dropna(subset=["review_count"])
      .sort_values("review_count", ascending=False)
      [["product", "review_count", "rating", "discount",
        "current_price_mid", "rating_category"]]
      .head(10)
)

top_engagement

,product,review_count,rating,discount,current_price_mid,rating_category
62,120W Cordless Vacuum Cleaners Handheld Electri...,69.0,2.8,0.49,445.0,Poor
6,137 Pieces Cake Decorating Tool Set Baking Sup...,55.0,4.6,0.24,2319.0,Excellent
25,Electronic Digital Display Vernier Caliper,49.0,4.6,0.35,420.0,Excellent
19,3D Waterproof EVA Plastic Shower Curtain 1.8*2...,44.0,4.6,0.49,998.0,Excellent
11,100 Pcs Crochet Hook Tool Set Knitting Hook Se...,39.0,4.7,0.34,990.0,Excellent
37,Punch-free Great Load Bearing Bathroom Storage...,36.0,4.3,0.41,389.0,Average
34,53 Pieces/Set Yarn Knitting Crochet Hooks With...,32.0,4.5,0.27,1980.0,Average
2,Portable Mini Cordless Car Vacuum Cleaner - Blue,24.0,4.6,0.25,2199.0,Excellent
28,52 Pieces Cake Decorating Tool Set Gift Kit Ba...,20.0,4.1,0.30,1758.0,Average
33,53Pcs/Set Yarn Knitting Crochet Hooks With Bag...,20.0,4.7,0.27,1940.0,Excellent


In [61]:
#Top 10 products with the highest discount
top_discounted = (
    df.sort_values("calculated_discount_pct", ascending=False)
      [["product", "calculated_discount_pct", "discount_amount",
        "rating", "review_count", "current_price_mid"]]
      .head(10)
)

top_discounted

,product,calculated_discount_pct,discount_amount,rating,review_count,current_price_mid
109,6 In 1 Bottle Can Opener Multifunctional Easy ...,0.640,354.0,NaN,NaN,199.0
48,Creative Owl Shape Keychain Black,0.605,305.0,NaN,NaN,199.0
58,Simple Metal Dog Art Sculpture Decoration For ...,0.555,497.0,NaN,NaN,399.0
61,5-PCS Stainless Steel Cooking Pot Set With Ste...,0.550,2585.0,2.1,13.0,2115.0
24,LASA 3 Tier Bamboo Shoe Bench Storage Shelf,0.545,2452.0,4.3,7.0,2048.0
8,LASA FOLDING TABLE SERVING STAND,0.545,1526.0,4.8,5.0,1274.0
60,Mythco 120COB Solar Wall Ligt With Motion Sens...,0.535,528.0,3.0,10.0,458.0
36,Classic Black Cat Cotton Hemp Pillow Case For ...,0.525,189.0,5.0,2.0,171.0
20,3PCS Single Head Knitting Crochet Sweater Need...,0.525,42.0,3.3,13.0,38.0
63,Intelligent LED Body Sensor Wireless Lighting...,0.522,355.0,2.7,15.0,325.0


In [62]:
# Calculate the thresholds from the dataset
high_discount = df["calculated_discount_pct"].quantile(0.75)
low_rating = df["rating"].quantile(0.25)
high_price = df["current_price_mid"].quantile(0.75)

pricing_review = (
    df[
        (df["calculated_discount_pct"] >= high_discount) &
        (
            (df["rating"] <= low_rating) |
            (df["current_price_mid"] >= high_price)
        )
    ]
    [["product", "calculated_discount_pct", "rating", "review_count",
      "current_price_mid", "discount_amount", "rating_category"]]
    .sort_values(
        ["calculated_discount_pct", "current_price_mid"],
        ascending=[False, False]
    )
)

pricing_review

,product,calculated_discount_pct,rating,review_count,current_price_mid,discount_amount,rating_category
61,5-PCS Stainless Steel Cooking Pot Set With Ste...,0.550,2.1,13.0,2115.0,2585.0,Poor
24,LASA 3 Tier Bamboo Shoe Bench Storage Shelf,0.545,4.3,7.0,2048.0,2452.0,Average
60,Mythco 120COB Solar Wall Ligt With Motion Sens...,0.535,3.0,10.0,458.0,528.0,Average
63,Intelligent LED Body Sensor Wireless Lighting...,0.522,2.7,15.0,325.0,355.0,Poor
66,380ML USB Rechargeable Portable Small Blenders...,0.500,2.3,7.0,1000.0,1000.0,Poor
110,Wall-mounted Sticker Punch-free Plug Fixer,0.500,2.0,1.0,450.0,450.0,Poor
17,LASA Aluminum Folding Truck Hand Cart - 68kg Max,0.490,5.0,3.0,2025.0,1946.0,Excellent
62,120W Cordless Vacuum Cleaners Handheld Electri...,0.490,2.8,69.0,445.0,428.0,Poor


In [63]:
print("High discount threshold:", high_discount)
print("Low rating threshold:", low_rating)
print("High price threshold:", high_price)

High discount threshold: 0.49
Low rating threshold: 3.0
High price threshold: 1669.5
